# Notebook 06 — AnomalyDINO: Multi-Class vs Single-Class Validation on MVTec AD

## Purpose

This notebook investigates whether AnomalyDINO's failure on Real-IAD is caused by
the multi-class evaluation protocol specifically, or by Real-IAD's dataset characteristics.

AnomalyDINO is evaluated on MVTec AD under two protocols:

1. **Single-class protocol** — one memory bank per category, 16 shots per category
   (3 runs, results averaged). This matches the model's primary published evaluation setting.
2. **Multi-class protocol** — one shared memory bank across all 15 categories, 16 shots
   per category (240 total reference images). This mirrors the Real-IAD evaluation protocol.

If performance degrades substantially under the multi-class protocol on MVTec AD — a dataset
where the model is known to work well in the single-class setting — this confirms that the
memory-based paradigm itself is sensitive to multi-class conditions, independent of dataset difficulty.

**Note:** Anomaly maps are not saved in this notebook. The focus is on image-level I-AUROC
for paradigm validation rather than localisation analysis.

## References
- Damm et al. (2025): AnomalyDINO — Boosting Patch-Based Few-Shot Anomaly Detection with DINOv2
- Bergmann et al. (2019): MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
repo_path = '/content/drive/MyDrive/BachelorsThesis'
results_path = f'{repo_path}/results'

import sys
sys.path.insert(0, repo_path)

print(f'Repo path: {repo_path}')
print(f'Results path: {results_path}')

In [ ]:
!pip install colorama -q

import os
if not os.path.exists('/content/datasets/MVTec/bottle'):
    print('Downloading MVTec AD dataset...')
    from anomalib.data import MVTecAD
    dm = MVTecAD(root='/content/datasets/MVTec', category='bottle')
    dm.prepare_data()
    print('Download complete')
else:
    print('MVTec AD dataset already present')

In [ ]:
import torch
import pandas as pd
import numpy as np
import gc
from sklearn.metrics import roc_auc_score
from torchvision.transforms import v2 as T
from anomalib.models import AnomalyDINO
from anomalib.data import MVTecAD

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

MVTEC_CATS = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill',
    'screw', 'tile', 'toothbrush', 'transistor',
    'wood', 'zipper'
]

# Preprocessing matching all three thesis models
# 448 resize -> 392 centre crop -> ImageNet normalisation
transform = T.Compose([
    T.Resize((448, 448)),
    T.CenterCrop(392),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

N_SHOTS = 16  # reference images per category

print('All imports loaded')

## 1. Single-Class Protocol (Baseline)

One memory bank per category, 16 reference images per category.
This matches AnomalyDINO's primary published evaluation setting.
Three runs with different reference image sets are averaged to reduce
variance from reference sample selection (Run 1: images 1-16,
Run 2: images 17-32, Run 3: images 33-48).

Published reference (672px, ViT-Small): 98.4% I-AUROC on MVTec AD (Hofer et al., 2025).
This experiment uses 448px and ViT-Base for consistency with the thesis backbone.

In [ ]:
def build_single_class_model(ref_images):
    model = AnomalyDINO(
        encoder_name='dinov2reg_vit_base_14',
        coreset_subsampling=False,
        masking=False,
    )
    tm = model.model.to(DEVICE)
    tm.train()
    with torch.no_grad():
        tm(ref_images.to(DEVICE))
    tm.embedding_store = [e.cpu() for e in tm.embedding_store]
    torch.cuda.empty_cache()
    tm.to('cpu')
    tm.fit()
    tm.to(DEVICE)
    tm.memory_bank = tm.memory_bank.to(DEVICE)
    model.model = tm
    return model

sc_run_results = {cat: [] for cat in MVTEC_CATS}

for run_idx in range(3):
    save_maps = (run_idx == 0)
    print(f'\nRun {run_idx + 1}/3 — maps: {save_maps}')
    for cat in MVTEC_CATS:
        datamodule = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            train_batch_size=64,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform,
        )
        datamodule.setup()
        all_train = []
        for b in datamodule.train_dataloader():
            all_train.append(b['image'])
        all_train = torch.cat(all_train, dim=0)
        start = run_idx * N_SHOTS
        ref_images = all_train[start:start + N_SHOTS]
        model = build_single_class_model(ref_images)

        if save_maps:
            cat_map_dir = f'{maps_mvtec_sc}/{cat}'
            os.makedirs(cat_map_dir, exist_ok=True)

        scores, labels = [], []
        model.model.eval()
        with torch.no_grad():
            for batch in datamodule.test_dataloader():
                out = model(batch['image'].to(DEVICE))
                scores.extend(out.pred_score.cpu().numpy().flatten())
                labels.extend(batch.gt_label.cpu().numpy().flatten())
                if save_maps:
                    for amap in out.anomaly_map.cpu().numpy():
                        idx = len(os.listdir(cat_map_dir))
                        np.save(f'{cat_map_dir}/{cat}_{idx:04d}.npy', amap)

        auroc = roc_auc_score(labels, scores) * 100
        sc_run_results[cat].append(auroc)
        print(f'  {cat}: {auroc:.2f}%')
        del model
        torch.cuda.empty_cache()
        gc.collect()

sc_rows = []
for cat in MVTEC_CATS:
    runs = sc_run_results[cat]
    sc_rows.append({
        'Category': cat,
        'Run 1': round(runs[0], 2),
        'Run 2': round(runs[1], 2),
        'Run 3': round(runs[2], 2),
        'Single-class I-AUROC': round(np.mean(runs), 2),
        'SC Std': round(np.std(runs), 2),
    })

sc_df = pd.DataFrame(sc_rows)
print(f'\nSingle-class mean I-AUROC: {sc_df["Single-class I-AUROC"].mean():.2f}%')
print(sc_df[['Category', 'Single-class I-AUROC', 'SC Std']].to_string(index=False))
sc_df.to_csv(f'{results_path}/mvtec_anomalydino_singleclass.csv', index=False)
print('\nSaved: mvtec_anomalydino_singleclass.csv')

## 2. Multi-Class Protocol

One shared memory bank across all 15 categories, 16 reference images per category
(240 total reference images). This mirrors the Real-IAD standard protocol where
all categories share a single memory bank.

In [ ]:
def build_single_class_model(ref_images):
    model = AnomalyDINO(
        encoder_name='dinov2reg_vit_base_14',
        coreset_subsampling=False,
        masking=False,
    )
    tm = model.model.to(DEVICE)
    tm.train()
    with torch.no_grad():
        tm(ref_images.to(DEVICE))
    tm.embedding_store = [e.cpu() for e in tm.embedding_store]
    torch.cuda.empty_cache()
    tm.to('cpu')
    tm.fit()
    tm.to(DEVICE)
    tm.memory_bank = tm.memory_bank.to(DEVICE)
    model.model = tm
    return model

sc_run_results = {cat: [] for cat in MVTEC_CATS}

for run_idx in range(3):
    save_maps = (run_idx == 0)
    print(f'\nRun {run_idx + 1}/3 — maps: {save_maps}')
    for cat in MVTEC_CATS:
        datamodule = MVTecAD(
            root='/content/datasets/MVTec',
            category=cat,
            train_batch_size=64,
            eval_batch_size=32,
            num_workers=2,
            augmentations=transform,
        )
        datamodule.setup()
        all_train = []
        for b in datamodule.train_dataloader():
            all_train.append(b['image'])
        all_train = torch.cat(all_train, dim=0)
        start = run_idx * N_SHOTS
        ref_images = all_train[start:start + N_SHOTS]
        model = build_single_class_model(ref_images)

        if save_maps:
            cat_map_dir = f'{maps_mvtec_sc}/{cat}'
            os.makedirs(cat_map_dir, exist_ok=True)

        scores, labels = [], []
        model.model.eval()
        with torch.no_grad():
            for batch in datamodule.test_dataloader():
                out = model(batch['image'].to(DEVICE))
                scores.extend(out.pred_score.cpu().numpy().flatten())
                labels.extend(batch.gt_label.cpu().numpy().flatten())
                if save_maps:
                    for amap in out.anomaly_map.cpu().numpy():
                        idx = len(os.listdir(cat_map_dir))
                        np.save(f'{cat_map_dir}/{cat}_{idx:04d}.npy', amap)

        auroc = roc_auc_score(labels, scores) * 100
        sc_run_results[cat].append(auroc)
        print(f'  {cat}: {auroc:.2f}%')
        del model
        torch.cuda.empty_cache()
        gc.collect()

sc_rows = []
for cat in MVTEC_CATS:
    runs = sc_run_results[cat]
    sc_rows.append({
        'Category': cat,
        'Run 1': round(runs[0], 2),
        'Run 2': round(runs[1], 2),
        'Run 3': round(runs[2], 2),
        'Single-class I-AUROC': round(np.mean(runs), 2),
        'SC Std': round(np.std(runs), 2),
    })

sc_df = pd.DataFrame(sc_rows)
print(f'\nSingle-class mean I-AUROC: {sc_df["Single-class I-AUROC"].mean():.2f}%')
print(sc_df[['Category', 'Single-class I-AUROC', 'SC Std']].to_string(index=False))
sc_df.to_csv(f'{results_path}/mvtec_anomalydino_singleclass.csv', index=False)
print('\nSaved: mvtec_anomalydino_singleclass.csv')

## 3. Comparison Summary

In [ ]:
comparison = mc_df.merge(
    sc_df[['Category', 'Single-class I-AUROC', 'SC Std']],
    on='Category'
)
comparison['Difference (MC - SC)'] = (
    comparison['Multi-class I-AUROC'] - comparison['Single-class I-AUROC']
).round(2)

print('=' * 72)
print('AnomalyDINO: Multi-class vs Single-class on MVTec AD (16-shot, 448px)')
print('=' * 72)
print(comparison[['Category', 'Single-class I-AUROC', 'SC Std',
                   'Multi-class I-AUROC', 'Difference (MC - SC)']].to_string(index=False))
print()
print(f'Mean single-class I-AUROC: {comparison["Single-class I-AUROC"].mean():.2f}%')
print(f'Mean multi-class I-AUROC:  {comparison["Multi-class I-AUROC"].mean():.2f}%')
print(f'Mean degradation (MC-SC):  {comparison["Difference (MC - SC)"].mean():.2f}pp')
print()
print('Published reference (672px, ViT-Small): 98.4% I-AUROC (Hofer et al., 2025)')

comparison.to_csv(
    f'{results_path}/mvtec_anomalydino_multiclass_comparison.csv',
    index=False)
print('\nSaved: mvtec_anomalydino_multiclass_comparison.csv')

deg = comparison['Difference (MC - SC)'].mean()
print('\nInterpretation:')
if abs(deg) > 20:
    print(f'  Severe degradation of {deg:.1f}pp under multi-class protocol.')
    print('  The memory-based paradigm is highly sensitive to multi-class conditions.')
    print('  AnomalyDINO failure on Real-IAD is consistent with this paradigm limitation')
    print('  and is not solely explained by Real-IAD dataset characteristics.')
elif abs(deg) > 5:
    print(f'  Moderate degradation of {deg:.1f}pp under multi-class protocol.')
else:
    print(f'  Minimal degradation — multi-class does not significantly affect performance.')